# 2. Modeling & Evaluation

## Overview
This notebook trains supervised and unsupervised models, evaluates performance, and applies model interpretability techniques.

**Models Trained:** Baseline, Ridge, Random Forest, XGBoost, LightGBM, SVM  
**Unsupervised:** KMeans, DBSCAN, PCA, t-SNE  
**Interpretability:** SHAP, Partial Dependence Plots, Coefficient Analysis

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.train_supervised import (
    train_baseline_model, train_linear_model,
    train_tree_ensemble, train_svm,
    train_all_models, save_model
)
from src.evaluate import (
    evaluate_regression_model, compare_models,
    generate_evaluation_report, get_best_model
)
from src.train_unsupervised import (
    perform_kmeans_clustering, perform_dbscan_clustering,
    perform_pca, perform_tsne,
    interpret_clusters, plot_elbow_and_silhouette
)
from src.utils import setup_logging, save_figure

# Setup
logger = setup_logging()
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

FIGURES_PATH = Path('../figures/')
MODELS_PATH = Path('../models/')
REPORTS_PATH = Path('../reports/')

for p in [FIGURES_PATH, MODELS_PATH, REPORTS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print('Setup complete!')

## 2.1 Load Preprocessed Data

Loading the preprocessed data saved from Notebook 01.

In [ ]:
# Load preprocessed data
processed_path = Path('../data/processed/')

X_train = np.load(processed_path / 'X_train.npy')
X_val = np.load(processed_path / 'X_val.npy')
X_test = np.load(processed_path / 'X_test.npy')
y_train = pd.read_csv(processed_path / 'y_train.csv').squeeze()
y_val = pd.read_csv(processed_path / 'y_val.csv').squeeze()
y_test = pd.read_csv(processed_path / 'y_test.csv').squeeze()
feature_names = joblib.load(processed_path / 'feature_names.joblib')
preprocessor = joblib.load(processed_path / 'preprocessor.joblib')

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feature_names)}')

---

## Part A: Supervised Learning

### 2.2 Train All Models

Training multiple models with hyperparameter tuning via cross-validation.

In [ ]:
# Train all models
task_type = 'regression'

print('Training all models (this may take a few minutes)...')
print('=' * 60)

all_models = train_all_models(
    X_train, y_train,
    task_type=task_type,
    cv=5,
    random_state=42
)

print(f'\nTrained {len(all_models)} models successfully.')

### 2.3 Model Comparison

Evaluating all models on the test set and comparing performance.

In [ ]:
# Extract models from (model, info) tuples
models_dict = {name: model for name, (model, info) in all_models.items()}

# Compare on test set
comparison_df = compare_models(
    models_dict, X_test, y_test, task_type
)

print('\n' + '=' * 60)
print('MODEL COMPARISON (Test Set)')
print('=' * 60)
comparison_df.round(4)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_to_plot = ['RMSE', 'MAE', 'R2']
colors = plt.cm.Set2(np.linspace(0, 1, len(comparison_df)))

for idx, metric in enumerate(metrics_to_plot):
    if metric in comparison_df.columns:
        values = comparison_df[metric]
        bars = axes[idx].bar(values.index, values.values, color=colors)
        axes[idx].set_title(f'{metric} by Model', fontsize=13, fontweight='bold')
        axes[idx].set_ylabel(metric, fontsize=11)
        axes[idx].tick_params(axis='x', rotation=45, labelsize=9)
        
        for bar, val in zip(bars, values.values):
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                          f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Supervised Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'model_comparison', FIGURES_PATH)
plt.show()

In [ ]:
# Save best model
best_name, best_model = get_best_model(models_dict, comparison_df, task_type)
print(f'\nBest model: {best_name}')
print(f'Best RMSE: {comparison_df.loc[best_name, "RMSE"]:.4f}')
print(f'Best R2: {comparison_df.loc[best_name, "R2"]:.4f}')

save_model(best_model, MODELS_PATH / 'best_model.joblib', {
    'name': best_name,
    'metrics': comparison_df.loc[best_name].to_dict()
})

# Generate report
generate_evaluation_report(comparison_df, REPORTS_PATH, task_type)
print('\nSaved best model and evaluation report.')

### 2.4 Prediction Error Analysis

Analyzing residuals for the best model to understand prediction patterns.

In [ ]:
# Residual analysis for best model
y_pred = best_model.predict(X_test)
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.5, s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual', fontsize=11)
axes[0].set_ylabel('Predicted', fontsize=11)
axes[0].set_title('Actual vs Predicted', fontsize=13, fontweight='bold')
axes[0].legend()

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].set_xlabel('Residual', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].axvline(0, color='red', linestyle='--')

# Residual vs Predicted
axes[2].scatter(y_pred, residuals, alpha=0.5, s=15)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('Predicted', fontsize=11)
axes[2].set_ylabel('Residual', fontsize=11)
axes[2].set_title('Residuals vs Predicted', fontsize=13, fontweight='bold')

plt.suptitle(f'{best_name} — Residual Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'residual_analysis', FIGURES_PATH)
plt.show()

---

## Part B: Unsupervised Learning

### 2.5 KMeans Clustering

Using elbow method and silhouette analysis to find optimal number of clusters.

In [ ]:
# KMeans clustering
print('Performing KMeans clustering...')
kmeans_results = perform_kmeans_clustering(
    X_train, k_range=list(range(2, 11)), random_state=42
)

# Plot elbow and silhouette
fig = plot_elbow_and_silhouette(kmeans_results)
save_figure(fig, 'kmeans_elbow_silhouette', FIGURES_PATH)
plt.show()

print(f'Optimal K: {kmeans_results["optimal_k"]}')
print(f'Best silhouette score: {max(kmeans_results["silhouette_scores"]):.4f}')

In [ ]:
# Cluster interpretation
cluster_stats, distinguishing = interpret_clusters(
    X_train, kmeans_results['labels'], feature_names[:X_train.shape[1]]
)

print('\nDistinguishing features per cluster:')
for cluster, features in distinguishing.items():
    print(f'\n  Cluster {cluster}:')
    for feat, zscore in list(features.items())[:3]:
        print(f'    {feat}: z-score = {zscore:.2f}')

### 2.6 DBSCAN Clustering

In [ ]:
# DBSCAN clustering
print('Performing DBSCAN clustering...')
dbscan_results = perform_dbscan_clustering(X_train)

print(f'\nBest params: {dbscan_results["best_params"]}')
print(f'Best silhouette: {dbscan_results["best_score"]:.4f}')

if dbscan_results['labels'] is not None:
    unique_labels = np.unique(dbscan_results['labels'])
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    n_noise = (dbscan_results['labels'] == -1).sum()
    print(f'Clusters found: {n_clusters}')
    print(f'Noise points: {n_noise}')

### 2.7 PCA (Dimensionality Reduction)

In [ ]:
# PCA
pca_model, X_pca = perform_pca(X_train, variance_threshold=0.95)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance per component
axes[0].bar(range(1, len(pca_model.explained_variance_ratio_) + 1),
            pca_model.explained_variance_ratio_, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Principal Component', fontsize=11)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=11)
axes[0].set_title('Explained Variance by Component', fontsize=13, fontweight='bold')

# Cumulative variance
cumsum = np.cumsum(pca_model.explained_variance_ratio_)
axes[1].plot(range(1, len(cumsum) + 1), cumsum, 'bo-', linewidth=2)
axes[1].axhline(0.95, color='r', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=11)
axes[1].set_title('Cumulative Explained Variance', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
save_figure(fig, 'pca_variance', FIGURES_PATH)
plt.show()

print(f'\nComponents for 95% variance: {pca_model.n_components_}')
print(f'Total variance explained: {cumsum[-1]:.2%}')

### 2.8 t-SNE Visualization

In [ ]:
# t-SNE
print('Performing t-SNE (this may take a moment)...')
X_tsne = perform_tsne(X_train, n_components=2, perplexity=30, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by target (using training target)
n_samples = min(len(X_tsne), len(y_train))
scatter1 = axes[0].scatter(
    X_tsne[:n_samples, 0], X_tsne[:n_samples, 1],
    c=y_train.values[:n_samples], cmap='viridis', alpha=0.5, s=10
)
plt.colorbar(scatter1, ax=axes[0], label='SalePrice')
axes[0].set_title('t-SNE (colored by target)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# Color by KMeans clusters
cluster_labels = kmeans_results['labels'][:n_samples]
scatter2 = axes[1].scatter(
    X_tsne[:n_samples, 0], X_tsne[:n_samples, 1],
    c=cluster_labels, cmap='Set1', alpha=0.5, s=10
)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')
axes[1].set_title('t-SNE (colored by KMeans cluster)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
save_figure(fig, 'tsne_visualization', FIGURES_PATH)
plt.show()

---

## Part C: Model Interpretability

### 2.9 SHAP Analysis

Using SHAP (SHapley Additive exPlanations) to understand feature contributions.

In [ ]:
import shap

# Use TreeExplainer for tree-based models, KernelExplainer for others
if best_name in ['RandomForest', 'XGBoost', 'LightGBM']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test[:100])
else:
    # For non-tree models, use a sample of training data
    background = shap.sample(X_train, 50)
    explainer = shap.KernelExplainer(best_model.predict, background)
    shap_values = explainer.shap_values(X_test[:50])

print(f'SHAP values computed for {best_name}')
print(f'Shape: {np.array(shap_values).shape}')

In [ ]:
# SHAP Summary Plot (Global Feature Importance)
n_features_used = min(len(feature_names), X_test.shape[1])
display_names = feature_names[:n_features_used]

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test[:len(shap_values)],
    feature_names=display_names,
    show=False,
    max_display=20
)
plt.title(f'SHAP Feature Importance ({best_name})', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(plt.gcf(), 'shap_summary', FIGURES_PATH)
plt.show()

In [ ]:
# SHAP Bar Plot (Mean absolute SHAP values)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test[:len(shap_values)],
    feature_names=display_names,
    plot_type='bar',
    show=False,
    max_display=20
)
plt.title(f'Mean |SHAP| Feature Importance ({best_name})', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(plt.gcf(), 'shap_bar', FIGURES_PATH)
plt.show()

In [ ]:
# SHAP Force Plot for individual predictions
print('Single prediction explanation (sample 0):')
print(f'  Actual: {y_test.iloc[0]:.2f}')
print(f'  Predicted: {best_model.predict(X_test[:1])[0]:.2f}')

shap.initjs()
shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[0],
    shap_values[0],
    feature_names=display_names,
    matplotlib=True
)
save_figure(plt.gcf(), 'shap_force_plot', FIGURES_PATH)
plt.show()

### 2.10 Partial Dependence Plots

Showing how individual features affect predictions.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Use top features from SHAP analysis
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_indices = np.argsort(mean_abs_shap)[-4:][::-1]  # Top 4 features

if best_name in ['RandomForest', 'XGBoost', 'LightGBM']:
    fig, ax = plt.subplots(figsize=(16, 10))
    PartialDependenceDisplay.from_estimator(
        best_model, X_test,
        features=top_indices.tolist(),
        feature_names=display_names,
        ax=ax
    )
    plt.suptitle(f'Partial Dependence Plots ({best_name})', fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'partial_dependence', FIGURES_PATH)
    plt.show()
else:
    print('Partial dependence plots are most useful for tree-based models.')

### 2.11 Linear Model Coefficient Analysis

In [ ]:
# Coefficient analysis for the linear model
linear_model = models_dict.get('Linear')

if linear_model is not None and hasattr(linear_model, 'coef_'):
    n_coefs = min(len(linear_model.coef_), len(feature_names))
    coef_df = pd.DataFrame({
        'Feature': feature_names[:n_coefs],
        'Coefficient': linear_model.coef_[:n_coefs]
    })
    coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
    coef_df = coef_df.sort_values('Abs_Coef', ascending=False)
    
    # Plot top 15 coefficients
    top_coefs = coef_df.head(15)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['green' if c > 0 else 'red' for c in top_coefs['Coefficient']]
    ax.barh(range(len(top_coefs)), top_coefs['Coefficient'], color=colors, alpha=0.7)
    ax.set_yticks(range(len(top_coefs)))
    ax.set_yticklabels(top_coefs['Feature'])
    ax.set_xlabel('Coefficient Value', fontsize=11)
    ax.set_title('Top 15 Linear Model Coefficients', fontsize=14, fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.5)
    ax.invert_yaxis()
    plt.tight_layout()
    save_figure(fig, 'linear_coefficients', FIGURES_PATH)
    plt.show()
else:
    print('Linear model not available or has no coefficients.')

---

## 2.12 Summary & Conclusions

### Supervised Learning Results
- Trained and compared 6 models: Baseline, Ridge, Random Forest, XGBoost, LightGBM, SVM
- All models significantly outperform the baseline
- Tree ensemble methods generally perform best for this regression task

### Unsupervised Learning Insights
- KMeans identified natural groupings in the housing data
- PCA showed dimensionality can be significantly reduced while retaining 95% variance
- t-SNE visualization reveals structure in the data aligned with price levels

### Model Interpretability
- SHAP analysis reveals the most important features driving predictions
- Partial dependence plots show non-linear relationships in tree models
- Linear model coefficients provide direct interpretability